# 09. Joint Flow Matching: path, velocity и учебное обучение

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 2](https://diffusion.csail.mit.edu/2026/docs/20260122_Lecture_02.pdf).
<!-- study-navigation:end -->

**Результат:** вывести target в noise→data sampling convention и обучить настоящий tiny joint CrossDiT

**Перед началом:** Lecture 2 / Lab 2; K6 08. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/06_dit_and_flow_matching.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §4.2](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/dit.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/dit.py)
- [`core/algo/flow_matching.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/flow_matching.py)
- [`core/algo/denoise_loop.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/denoise_loop.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','main'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '09_flow_matching_training'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Сначала соглашение о времени

В этом опыте t=0 — data, t=1 — noise; sampler идёт от 1 к 0.

$$
x_t=(1-t)x_0+t\epsilon,\qquad u^*(x_t,t\mid x_0,\epsilon)=\epsilon-x_0.
$$

Target задан для каждой пары endpoints. Модель видит только current state, time и condition;
минимум MSE соответствует усреднённому conditional target при данном current state.

$$
\mathcal L=\lambda_v\mathbb E\|u_\theta^v-u_v^*\|^2+
\lambda_a\mathbb E\|u_\theta^a-u_a^*\|^2.
$$

Модальности могут иметь разные t. `model_time=1000*t` — input convention сети,
но Euler update использует разности **нормированного t**, без дополнительного множителя 1000.
[Flow Matching](https://arxiv.org/abs/2210.02747) и
[Rectified Flow](https://arxiv.org/abs/2209.03003) — первичные математические источники.


In [ ]:
x0=torch.randn(4,3);eps=torch.randn_like(x0);t=torch.tensor(.4);h=1e-3
path=lambda time:(1-time)*x0+time*eps
fd=(path(t+h)-path(t-h))/(2*h);target=eps-x0
error=(fd-target).abs().max().item();assert error<3e-4
metrics['finite_difference_target_error']=error
print('dx/dt error:',error,'sampler dt sign:',float(torch.tensor(0.)-torch.tensor(1.)))


## 2. Данные с известной общей причиной

Создадим синтетические video/audio latents от scalar c; text embeddings также несут c.
Это позволяет учить conditioning без dataset/codec weights. Все данные — случайные числа,
не речь и не изображения. Веса внешних encoders здесь вообще не создаются.

Учебные lambda_v=lambda_a=0.5 относятся только к этому toy objective. Мы не воспроизводим
исторический industrial trainer, face loss, RL, model soup или distillation.
Native text projection cache очищается **перед каждым training forward**.


In [ ]:
model=tiny_model().train();_,_,template=tiny_inputs(model,batch=8)
opt=torch.optim.Adam(model.parameters(),lr=.002)
def batch(seed):
    g=torch.Generator().manual_seed(seed);B=8
    c=torch.randn(B,1,generator=g)
    v0=c.view(B,1,1,1,1)+.1*torch.randn(B,2,4,4,4,generator=g)
    a0=c.view(B,1,1)+.1*torch.randn(B,6,4,generator=g)
    ev=torch.randn(v0.shape,generator=g);ea=torch.randn(a0.shape,generator=g)
    tv=.1+.8*torch.rand(B,generator=g);ta=.1+.8*torch.rand(B,generator=g)
    vt=tv.view(B,1,1,1,1);at=ta.view(B,1,1)
    v=(1-vt)*v0+vt*ev;a=(1-at)*a0+at*ea
    kw=dict(template,x_video=torch.cat((v,torch.zeros_like(v),torch.zeros_like(v[...,:1])),-1),
            x_audio=a,text_embed=c[:,None,:].expand(B,5,12).clone(),
            pooled_text_embed=c.expand(B,8).clone(),time=[tv*1000,ta*1000])
    return kw,(ev-v0,ea-a0)
def losses(seed):
    kw,target=batch(seed);v,a=fresh_forward(model,kw)
    return F.mse_loss(v,target[0]),F.mse_loss(a,target[1])
with torch.no_grad():initial=[x.item() for x in losses(999)]
history=[]
for step in range(100):
    lv,la=losses(step+100);loss=.5*lv+.5*la
    opt.zero_grad();loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.)
    opt.step();history.append([lv.item(),la.item()])
with torch.no_grad():final=[x.item() for x in losses(999)]
assert np.isfinite(history).all() and sum(final)<sum(initial)
metrics['heldout_initial_video_audio']=initial;metrics['heldout_final_video_audio']=final
metrics['trainable_parameters']=sum(p.numel() for p in model.parameters() if p.requires_grad)
fig,ax=plt.subplots(figsize=(8,3));ax.plot(np.array(history));ax.legend(['video MSE','audio MSE'])
ax.set_xlabel('Toy training step');ax.set_ylabel('MSE');fig.tight_layout()
fig.savefig(figure_path(LAB,'joint_fm_training'));plt.show();print('Held-out:',initial,'->',final)


## 3. Почему шумы и времена требуют отдельного контроля

Сходные значения MSE двух ветвей ещё не означают одинаковое perceptual качество:
одна ошибка измерена на 16-channel video latents, другая — на 40-channel audio latents,
масштаб и размерность зависят от codecs. При смене codec scale velocity target тоже меняется.
Для полной модели нужны waveform/video metrics и resource logs; toy validation проверяет только обучение функции.


## Самостоятельное исследование

1. Сравните shared t и independent t при фиксированных data/seeds/optimizer.
2. Заморозьте audio backbone и fusion: что останется trainable и какие loss имеют gradient?
3. Смените время s=1-t и выведите знак нового velocity, прежде чем менять код sampler.
4. Почему endpoint x0 можно использовать в target при training, но нельзя передавать sampler как неизвестное условие?

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы можете выписать training batch и optimizer contract, назвать model inputs и target,
а также объяснить, почему маленький native CrossDiT учится без загрузки pretrained weights.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 08. Dual-stream CrossDiT K6: два потока Audio + Video](08_dual_stream_audio_video_crossdit.ipynb) · [10. Lite Distill: grid-head, DX policy и π-Flow →](10_piflow_distillation.ipynb)
